In [5]:
"""
01_simulate_customers.py
────────────────────────
Generates: customers.csv
Rows:       2,000
Table:      Master customer table — one row per customer
"""

'\n01_simulate_customers.py\n────────────────────────\nGenerates: customers.csv\nRows:       2,000\nTable:      Master customer table — one row per customer\n'

In [6]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os

In [7]:
# ── Reproducibility ───────────────────────────────────────────────
# Fixed seed ensures every team member gets identical data
np.random.seed(42)

In [8]:
# ── Config ────────────────────────────────────────────────────────
NUM_CUSTOMERS = 2000
START_DATE    = datetime(2023, 1, 1)
END_DATE      = datetime(2024, 12, 31)
TOTAL_DAYS    = (END_DATE - START_DATE).days   # 730 days

In [9]:
# Plan distribution: most users pick cheapest plan (realistic SaaS behaviour)
PLANS    = ['Starter', 'Pro', 'Enterprise']
PLAN_P   = [0.50,      0.35,  0.15]
PLAN_MRR = {'Starter': 29, 'Pro': 99, 'Enterprise': 399}

In [10]:
# Acquisition channels with realistic weight
CHANNELS  = ['Organic', 'Paid Search', 'Social', 'Referral', 'Email', 'Direct']
CHANNEL_P = [0.30,       0.25,          0.15,     0.15,       0.10,    0.05]

In [11]:
# Country distribution — US-heavy, global spread
COUNTRIES  = ['USA', 'UK', 'India', 'Canada', 'Australia', 'Germany', 'France', 'Other']
COUNTRY_P  = [0.45,  0.12, 0.10,    0.08,      0.06,        0.07,      0.05,     0.07]

In [12]:
#── Simulate signup dates with a growth curve ─────────────────────
# np.random.exponential gives a right-skewed distribution:
# fewer signups early (startup is new), more signups later (word spreads).
# This is more realistic than# a flat uniform distribution.
raw_days    = np.random.exponential(scale=TOTAL_DAYS * 0.6, size=NUM_CUSTOMERS)
clipped     = np.clip(raw_days, 0, TOTAL_DAYS - 1).astype(int)
signup_days = sorted(clipped)   # sort so early IDs = early customer

In [13]:
# ── Build rows ────────────────────────────────────────────────────
customers = []
for i, day_offset in enumerate(signup_days):
    plan    = np.random.choice(PLANS, p=PLAN_P)
    channel = np.random.choice(CHANNELS, p=CHANNEL_P)
    country = np.random.choice(COUNTRIES, p=COUNTRY_P)
    signup  = START_DATE + timedelta(days=int(day_offset))

    customers.append({
        'customer_id':         f'C{i+1:04d}',
        'signup_date':         signup.strftime('%Y-%m-%d'),
        'plan_type':           plan,
        'acquisition_channel': channel,
        'country':             country,
        'mrr':                 PLAN_MRR[plan]
    })

In [14]:
df = pd.DataFrame(customers)

In [15]:
# ── Data quality check ────────────────────────────────────────────
assert len(df) == NUM_CUSTOMERS,        "Row count mismatch"
assert df.isnull().sum().sum() == 0,    "Unexpected nulls found"
assert df['customer_id'].nunique() == NUM_CUSTOMERS, "Duplicate IDs found"
assert df['signup_date'].min() >= START_DATE.strftime('%Y-%m-%d'), "Date out of range"
assert df['signup_date'].max() <= END_DATE.strftime('%Y-%m-%d'),   "Date out of range"

In [16]:
# ── Save ──────────────────────────────────────────────────────────
os.makedirs('data', exist_ok=True)
df.to_csv('data/customers.csv', index=False)

In [17]:
# ── Summary ───────────────────────────────────────────────────────
print("=" * 45)
print("customers.csv — Quality Report")
print("=" * 45)
print(f"Rows          : {len(df):,}")
print(f"Null values   : {df.isnull().sum().sum()}")
print(f"Date range    : {df['signup_date'].min()} → {df['signup_date'].max()}")
print(f"\nPlan breakdown:\n{df['plan_type'].value_counts().to_string()}")
print(f"\nChannel breakdown:\n{df['acquisition_channel'].value_counts().to_string()}")
print(f"\nCountry breakdown:\n{df['country'].value_counts().to_string()}")
print(f"\nMRR stats:\n{df['mrr'].describe().to_string()}")
print(f"\nSample (first 5 rows):")
print(df.head(5).to_string(index=False))
print("\n✅ Saved: data/customers.csv")

customers.csv — Quality Report
Rows          : 2,000
Null values   : 0
Date range    : 2023-01-02 → 2024-12-30

Plan breakdown:
plan_type
Starter       1063
Pro            658
Enterprise     279

Channel breakdown:
acquisition_channel
Organic        590
Paid Search    488
Social         322
Referral       316
Email          191
Direct          93

Country breakdown:
country
USA          920
UK           257
India        183
Canada       143
Other        142
Germany      133
Australia    113
France       109

MRR stats:
count    2000.000000
mean      103.645000
std       123.066432
min        29.000000
25%        29.000000
50%        29.000000
75%        99.000000
max       399.000000

Sample (first 5 rows):
customer_id signup_date  plan_type acquisition_channel   country  mrr
      C0001  2023-01-02    Starter             Organic    France   29
      C0002  2023-01-03    Starter             Organic Australia   29
      C0003  2023-01-03    Starter            Referral       USA   29
   

In [18]:
"""
02_simulate_monthly_activity.py
────────────────────────────────
Depends on:  data/customers.csv  (run 01 first)
Generates:   data/monthly_activity.csv
Rows:        ~17,500 (varies — only months after each customer's signup)
Table:       One row per customer per active month — includes churn logic
 
Run:  python 02_simulate_monthly_activity.py
Output saved to: data/monthly_activity.csv
"""

"\n02_simulate_monthly_activity.py\n────────────────────────────────\nDepends on:  data/customers.csv  (run 01 first)\nGenerates:   data/monthly_activity.csv\nRows:        ~17,500 (varies — only months after each customer's signup)\nTable:       One row per customer per active month — includes churn logic\n\nRun:  python 02_simulate_monthly_activity.py\nOutput saved to: data/monthly_activity.csv\n"

In [19]:
import pandas as pd
import numpy as np
from datetime import datetime
from dateutil.relativedelta import relativedelta
import os

In [20]:
# ── Reproducibility ───────────────────────────────────────────────
np.random.seed(42)
 
# ── Load customers ────────────────────────────────────────────────
df_customers = pd.read_csv('data/customers.csv', parse_dates=['signup_date'])
print(f"Loaded customers.csv — {len(df_customers):,} rows")
 

Loaded customers.csv — 2,000 rows


In [21]:
# ── Config ────────────────────────────────────────────────────────
SIM_START = datetime(2023, 1, 1)
all_months = [SIM_START + relativedelta(months=m) for m in range(24)]
 
# Monthly churn probability by plan
# Enterprise = sticky (contracts, integrations, high switching cost)
# Starter    = price-sensitive, casual users churn more
MONTHLY_CHURN = {
    'Starter':    0.07,   # 7% chance of churning each month
    'Pro':        0.04,   # 4%
    'Enterprise': 0.02,   # 2%
}

In [22]:
# Sessions and features used per month, by plan
# Enterprise users are power users — higher engagement
SESSION_PARAMS = {
    'Starter':    {'sessions': (3,  10),  'features': (1, 4)},
    'Pro':        {'sessions': (8,  25),  'features': (3, 8)},
    'Enterprise': {'sessions': (20, 60),  'features': (6, 12)},
}

In [23]:
# ── Simulate activity for each customer across 24 months ──────────
records = []
 
for _, customer in df_customers.iterrows():
    cid         = customer['customer_id']
    plan        = customer['plan_type']
    signup      = customer['signup_date']
    churn_p     = MONTHLY_CHURN[plan]
    has_churned = False
 
    for month in all_months:
 
        # Skip months before this customer signed up
        if month < signup.replace(day=1):
            continue
 
        # Once a customer churns, they stay churned (no reactivation in this model)
        if has_churned:
            continue
        
        # Roll the churn dice for this month
        if np.random.random() < churn_p:
            has_churned   = True
            is_active     = False
            sessions      = 0
            features_used = 0
        else:
            is_active = True
            lo, hi    = SESSION_PARAMS[plan]['sessions']
            sessions  = np.random.randint(lo, hi + 1)
            lo, hi    = SESSION_PARAMS[plan]['features']
            features_used = np.random.randint(lo, hi + 1)
 
        records.append({
            'customer_id':   cid,
            'month':         month.strftime('%Y-%m-01'),
            'plan_type':     plan,
            'is_active':     is_active,
            'sessions':      sessions,
            'features_used': features_used,
        })

In [24]:
df_activity = pd.DataFrame(records)

In [25]:
# ── Data quality check ────────────────────────────────────────────
assert df_activity.isnull().sum().sum() == 0, "Unexpected nulls found"
assert df_activity['customer_id'].nunique() == len(df_customers), "Missing customers"
assert df_activity['month'].nunique() <= 24, "Too many months"

In [26]:
# ── Save ──────────────────────────────────────────────────────────
os.makedirs('data', exist_ok=True)
df_activity.to_csv('data/monthly_activity.csv', index=False)

In [27]:
# ── Summary ───────────────────────────────────────────────────────
mau = (df_activity[df_activity['is_active']]
       .groupby('month')['customer_id']
       .nunique()
       .reset_index()
       .rename(columns={'customer_id': 'MAU'}))
 
print("=" * 45)
print("monthly_activity.csv — Quality Report")
print("=" * 45)
print(f"Rows             : {len(df_activity):,}")
print(f"Null values      : {df_activity.isnull().sum().sum()}")
print(f"Unique customers : {df_activity['customer_id'].nunique():,}")
print(f"Months covered   : {df_activity['month'].nunique()}")
print(f"\nActive rate by plan:")
print(df_activity.groupby('plan_type')['is_active'].mean().round(3).to_string())
print(f"\nMAU trend (first 6 months):")
print(mau.head(6).to_string(index=False))
print(f"\nMAU trend (last 6 months):")
print(mau.tail(6).to_string(index=False))
print(f"\nSample (first 5 rows):")
print(df_activity.head(5).to_string(index=False))
print("\n✅ Saved: data/monthly_activity.csv")

monthly_activity.csv — Quality Report
Rows             : 17,493
Null values      : 0
Unique customers : 2,000
Months covered   : 24

Active rate by plan:
plan_type
Enterprise    0.977
Pro           0.958
Starter       0.934

MAU trend (first 6 months):
     month  MAU
2023-01-01  135
2023-02-01  247
2023-03-01  367
2023-04-01  439
2023-05-01  505
2023-06-01  562

MAU trend (last 6 months):
     month  MAU
2024-07-01  822
2024-08-01  810
2024-09-01  790
2024-10-01  785
2024-11-01  771
2024-12-01 1123

Sample (first 5 rows):
customer_id      month plan_type  is_active  sessions  features_used
      C0001 2023-01-01   Starter       True         7              3
      C0001 2023-02-01   Starter       True         7              1
      C0001 2023-03-01   Starter       True         5              3
      C0001 2023-04-01   Starter      False         0              0
      C0002 2023-01-01   Starter       True         6              4

✅ Saved: data/monthly_activity.csv


In [28]:
"""
03_simulate_subscriptions.py
─────────────────────────────
Depends on:  data/customers.csv        (run 01 first)
             data/monthly_activity.csv (run 02 first)
Generates:   data/subscriptions.csv
Rows:        2,000 (one per customer)
Table:       Subscription lifecycle — start date, churn date, MRR, upgrades
 
Note on NULLs: end_date is intentionally NULL for customers still active
               as of Dec 2024. This is NOT missing data — it means active.
 
Run:  python 03_simulate_subscriptions.py
Output saved to: data/subscriptions.csv
"""

'\n03_simulate_subscriptions.py\n─────────────────────────────\nDepends on:  data/customers.csv        (run 01 first)\n             data/monthly_activity.csv (run 02 first)\nGenerates:   data/subscriptions.csv\nRows:        2,000 (one per customer)\nTable:       Subscription lifecycle — start date, churn date, MRR, upgrades\n\nNote on NULLs: end_date is intentionally NULL for customers still active\n               as of Dec 2024. This is NOT missing data — it means active.\n\nRun:  python 03_simulate_subscriptions.py\nOutput saved to: data/subscriptions.csv\n'

In [29]:
import pandas as pd
import numpy as np
from datetime import datetime
from dateutil.relativedelta import relativedelta
import os
 
# ── Reproducibility ───────────────────────────────────────────────
np.random.seed(42)

In [30]:
# ── Load dependencies ─────────────────────────────────────────────
df_customers = pd.read_csv('data/customers.csv', parse_dates=['signup_date'])
df_activity  = pd.read_csv('data/monthly_activity.csv', parse_dates=['month'])
print(f"Loaded customers.csv      — {len(df_customers):,} rows")
print(f"Loaded monthly_activity.csv — {len(df_activity):,} rows")

Loaded customers.csv      — 2,000 rows
Loaded monthly_activity.csv — 17,493 rows


In [31]:
# ── Config ────────────────────────────────────────────────────────
PLAN_MRR  = {'Starter': 29, 'Pro': 99, 'Enterprise': 399}
SIM_END   = datetime(2024, 12, 1)   # Last month in simulation
 
# ── Derive churn date from activity table ─────────────────────────
# Find the last month each customer was active.
# If their last active month is Dec 2024 → still active (end_date = NULL)
# Otherwise → churned (end_date = month after last active month)
last_active = (
    df_activity[df_activity['is_active']]
    .groupby('customer_id')['month']
    .max()
    .reset_index()
    .rename(columns={'month': 'last_active_month'})
)

In [32]:
# Merge last active info into customers
subs = df_customers.merge(last_active, on='customer_id', how='left')
 
def get_end_date(row):
    last = row['last_active_month']
    # Customer was never active (edge case) → treat as churned immediately
    if pd.isna(last):
        return row['signup_date'].strftime('%Y-%m-%d')
    # Customer still active in Dec 2024 → no end date (NULL)
    if last >= SIM_END:
        return None
    # Churned → end date is the month after their last active month
    return (last + relativedelta(months=1)).strftime('%Y-%m-%d')
 
subs['start_date'] = subs['signup_date'].dt.strftime('%Y-%m-%d')
subs['end_date']   = subs.apply(get_end_date, axis=1)
subs['mrr']        = subs['plan_type'].map(PLAN_MRR)

In [33]:
# ~15% of customers upgraded their plan at least once during their tenure
subs['plan_changes'] = np.where(
    np.random.random(len(subs)) < 0.15,
    np.random.randint(1, 3, len(subs)),   # 1 or 2 upgrades
    0                                      # no change
)

In [34]:
# ── Select final columns ──────────────────────────────────────────
df_subs = subs[['customer_id', 'plan_type', 'start_date', 'end_date', 'mrr', 'plan_changes']].copy()

In [35]:
# ── Data quality check ────────────────────────────────────────────
assert len(df_subs) == len(df_customers), "Row count mismatch with customers table"
assert df_subs['customer_id'].nunique() == len(df_customers), "Duplicate customer IDs"
assert df_subs[['customer_id', 'plan_type', 'start_date', 'mrr']].isnull().sum().sum() == 0, \
    "Null found in non-nullable column"
# end_date nulls are expected and valid (= still active)

In [36]:
# ── Save ──────────────────────────────────────────────────────────
os.makedirs('data', exist_ok=True)
df_subs.to_csv('data/subscriptions.csv', index=False)

In [37]:
# ── Summary ───────────────────────────────────────────────────────
churned = df_subs['end_date'].notna().sum()
active  = df_subs['end_date'].isna().sum()
 
print("=" * 45)
print("subscriptions.csv — Quality Report")
print("=" * 45)
print(f"Rows                : {len(df_subs):,}")
print(f"Null in end_date    : {active:,}  ← intentional (still active)")
print(f"Churned customers   : {churned:,}")
print(f"Still active        : {active:,}")
print(f"Customers upgraded  : {(df_subs['plan_changes'] > 0).sum():,}")
print(f"\nMRR by plan:")
print(df_subs.groupby('plan_type')['mrr'].agg(['count', 'sum']).to_string())
print(f"\nSample (first 5 rows):")
print(df_subs.head(5).to_string(index=False))
print("\n✅ Saved: data/subscriptions.csv")

subscriptions.csv — Quality Report
Rows                : 2,000
Null in end_date    : 1,123  ← intentional (still active)
Churned customers   : 877
Still active        : 1,123
Customers upgraded  : 323

MRR by plan:
            count     sum
plan_type                
Enterprise    279  111321
Pro           658   65142
Starter      1063   30827

Sample (first 5 rows):
customer_id  plan_type start_date   end_date  mrr  plan_changes
      C0001    Starter 2023-01-02 2023-04-01   29             0
      C0002    Starter 2023-01-03 2023-12-01   29             0
      C0003    Starter 2023-01-03 2023-02-01   29             0
      C0004    Starter 2023-01-03       None   29             0
      C0005 Enterprise 2023-01-03       None  399             0

✅ Saved: data/subscriptions.csv


In [38]:
"""
04_simulate_marketing_spend.py
───────────────────────────────
Depends on:  nothing (standalone)
Generates:   data/marketing_spend.csv
Rows:        144  (24 months × 6 channels)
Table:       Monthly marketing spend and leads generated per channel
 
Run:  python 04_simulate_marketing_spend.py
Output saved to: data/marketing_spend.csv
"""

'\n04_simulate_marketing_spend.py\n───────────────────────────────\nDepends on:  nothing (standalone)\nGenerates:   data/marketing_spend.csv\nRows:        144  (24 months × 6 channels)\nTable:       Monthly marketing spend and leads generated per channel\n\nRun:  python 04_simulate_marketing_spend.py\nOutput saved to: data/marketing_spend.csv\n'

In [39]:
import pandas as pd
import numpy as np
from datetime import datetime
from dateutil.relativedelta import relativedelta
import os
 
# ── Reproducibility ───────────────────────────────────────────────
np.random.seed(42)

In [40]:
# ── Config ────────────────────────────────────────────────────────
SIM_START  = datetime(2023, 1, 1)
all_months = [SIM_START + relativedelta(months=m) for m in range(24)]
 
CHANNELS = ['Organic', 'Paid Search', 'Social', 'Referral', 'Email', 'Direct']
 
# Base monthly spend per channel (USD)
# Reflects realistic relative spend: Paid Search is most expensive,
# Email and Direct are cheap
BASE_SPEND = {
    'Organic':     500,    # SEO content creation, low direct cost
    'Paid Search': 4000,   # Google/Bing Ads — highest volume channel
    'Social':      2500,   # Facebook/Instagram/LinkedIn Ads
    'Referral':    800,    # Referral bonuses paid to advocates
    'Email':       300,    # Email platform + content costs
    'Direct':      200,    # Events, cold outreach, partnerships
}
 
# Leads generated per $1,000 spent — Referral is most efficient,
# Direct is least (low volume, high-touch)
LEADS_PER_1K = {
    'Organic':     8,
    'Paid Search': 5,
    'Social':      6,
    'Referral':    12,
    'Email':       10,
    'Direct':      4,
}

In [41]:
# ── Simulate rows ─────────────────────────────────────────────────
spend_rows = []
 
for m_idx, month in enumerate(all_months):
    # Budget grows ~2% per month as the startup scales and gains funding
    growth_factor = 1 + (m_idx * 0.02)
 
    for channel in CHANNELS:
        # Apply growth factor + small random noise (±10%)
        # Models month-to-month budget fluctuation realistically
        base  = BASE_SPEND[channel] * growth_factor
        noise = np.random.normal(1.0, 0.10)
        spend = round(max(base * noise, 100), 2)   # minimum $100/month
 
        # Leads generated = (spend / $1000) × efficiency rate × noise
        leads = int((spend / 1000) * LEADS_PER_1K[channel] * np.random.normal(1.0, 0.15))
        leads = max(leads, 1)   # at least 1 lead per channel per month
 
        spend_rows.append({
            'month':           month.strftime('%Y-%m-01'),
            'channel':         channel,
            'spend_amount':    spend,
            'leads_generated': leads,
        })

In [42]:
df_spend = pd.DataFrame(spend_rows)

In [43]:
# ── Data quality check ────────────────────────────────────────────
assert len(df_spend) == 144, f"Expected 144 rows, got {len(df_spend)}"
assert df_spend.isnull().sum().sum() == 0, "Unexpected nulls found"
assert df_spend['channel'].nunique() == 6, "Channel count mismatch"
assert df_spend['month'].nunique() == 24, "Month count mismatch"
assert (df_spend['spend_amount'] > 0).all(), "Zero or negative spend found"
assert (df_spend['leads_generated'] > 0).all(), "Zero leads found"
 
# ── Save ──────────────────────────────────────────────────────────
os.makedirs('data', exist_ok=True)
df_spend.to_csv('data/marketing_spend.csv', index=False)

In [44]:
# ── Summary ───────────────────────────────────────────────────────
print("=" * 45)
print("marketing_spend.csv — Quality Report")
print("=" * 45)
print(f"Rows             : {len(df_spend):,}")
print(f"Null values      : {df_spend.isnull().sum().sum()}")
print(f"Months covered   : {df_spend['month'].nunique()}")
print(f"Channels         : {df_spend['channel'].nunique()}")
print(f"\nTotal spend by channel:")
print(df_spend.groupby('channel')['spend_amount'].sum().sort_values(ascending=False).round(0).to_string())
print(f"\nTotal leads by channel:")
print(df_spend.groupby('channel')['leads_generated'].sum().sort_values(ascending=False).to_string())
print(f"\nOverall totals:")
print(f"  Total spend   : ${df_spend['spend_amount'].sum():,.2f}")
print(f"  Total leads   : {df_spend['leads_generated'].sum():,}")
blended_cpl = df_spend['spend_amount'].sum() / df_spend['leads_generated'].sum()
print(f"  Cost per lead : ${blended_cpl:.2f}")
print(f"\nSample (first 6 rows — Jan 2023 all channels):")
print(df_spend.head(6).to_string(index=False))
print("\n✅ Saved: data/marketing_spend.csv")

marketing_spend.csv — Quality Report
Rows             : 144
Null values      : 0
Months covered   : 24
Channels         : 6

Total spend by channel:
channel
Paid Search    112687.0
Social          73335.0
Referral        24589.0
Organic         14809.0
Email            8904.0
Direct           5790.0

Total leads by channel:
channel
Paid Search    550
Social         457
Referral       274
Organic         99
Email           76
Direct          24

Overall totals:
  Total spend   : $240,113.97
  Total leads   : 1,480
  Cost per lead : $162.24

Sample (first 6 rows — Jan 2023 all channels):
     month     channel  spend_amount  leads_generated
2023-01-01     Organic        524.84                4
2023-01-01 Paid Search       4259.08               26
2023-01-01      Social       2441.46               14
2023-01-01    Referral        926.34               12
2023-01-01       Email        285.92                3
2023-01-01      Direct        190.73                1

✅ Saved: data/marketing_spen

In [45]:
"""
05_simulate_revenue.py
───────────────────────
Depends on:  data/customers.csv        (run 01 first)
             data/monthly_activity.csv (run 02 first)
Generates:   data/revenue.csv
Rows:        24 (one per month)
Table:       MRR waterfall — new / expansion / churned / total MRR per month
 
The MRR waterfall is the most important financial summary in SaaS:
  total_mrr = prev_total + new_mrr + expansion_mrr - churned_mrr
 
Run:  python 05_simulate_revenue.py
Output saved to: data/revenue.csv
"""

'\n05_simulate_revenue.py\n───────────────────────\nDepends on:  data/customers.csv        (run 01 first)\n             data/monthly_activity.csv (run 02 first)\nGenerates:   data/revenue.csv\nRows:        24 (one per month)\nTable:       MRR waterfall — new / expansion / churned / total MRR per month\n\nThe MRR waterfall is the most important financial summary in SaaS:\n  total_mrr = prev_total + new_mrr + expansion_mrr - churned_mrr\n\nRun:  python 05_simulate_revenue.py\nOutput saved to: data/revenue.csv\n'

In [46]:
import pandas as pd
import numpy as np
from datetime import datetime
from dateutil.relativedelta import relativedelta
import os
 
# ── Reproducibility ───────────────────────────────────────────────
np.random.seed(42)

In [47]:
# ── Load dependencies ─────────────────────────────────────────────
df_customers = pd.read_csv('data/customers.csv', parse_dates=['signup_date'])
df_activity  = pd.read_csv('data/monthly_activity.csv', parse_dates=['month'])
print(f"Loaded customers.csv        — {len(df_customers):,} rows")
print(f"Loaded monthly_activity.csv — {len(df_activity):,} rows")

Loaded customers.csv        — 2,000 rows
Loaded monthly_activity.csv — 17,493 rows


In [48]:
# ── Config ────────────────────────────────────────────────────────
SIM_START  = datetime(2023, 1, 1)
all_months = [SIM_START + relativedelta(months=m) for m in range(24)]

In [49]:
# ── Pre-compute: signup month per customer ────────────────────────
df_customers['signup_month'] = (
    df_customers['signup_date']
    .dt.to_period('M')
    .dt.to_timestamp()
)

In [52]:
# ── Build revenue waterfall row by row ───────────────────────────
rev_rows      = []
running_total = 0.0
 
for month in all_months:
    month_ts = pd.Timestamp(month)
 
    # ── New MRR ──────────────────────────────────────────────────
    # Sum of MRR from customers who signed up this month
    new_custs = df_customers[df_customers['signup_month'] == month_ts]
    new_mrr   = float(new_custs['mrr'].sum())
 
    # ── Churned MRR ───────────────────────────────────────────────
    # MRR lost from customers who became inactive this month.
    # We find rows where is_active = False for this month,
    # then look up their MRR from the customers table.
    inactive_this_month = df_activity[
        (df_activity['month'] == month_ts) &
        (df_activity['is_active'] == False)
    ][['customer_id']]
 
    churned_mrr = float(
        inactive_this_month
        .merge(df_customers[['customer_id', 'mrr']], on='customer_id', how='left')['mrr']
        .sum()
    )

In [53]:
# ── Expansion MRR ─────────────────────────────────────────────
# Revenue from existing customers upgrading their plan.
# Modelled as: 5% of active customers generate ~$10 average upgrade value,
# with some random variance each month.
active_count = int(df_activity[
    (df_activity['month'] == month_ts) &
    (df_activity['is_active'])
].shape[0])
expansion_mrr = round(active_count * 0.05 * max(np.random.normal(10, 2), 0), 2)

# ── Total MRR (running cumulative) ────────────────────────────
running_total = running_total + new_mrr + expansion_mrr - churned_mrr
running_total = max(running_total, 0.0)   # can't go negative

rev_rows.append({
    'month':         month.strftime('%Y-%m-01'),
    'new_mrr':       round(new_mrr, 2),
    'expansion_mrr': round(expansion_mrr, 2),
    'churned_mrr':   round(churned_mrr, 2),
    'total_mrr':     round(running_total, 2),
})

df_revenue = pd.DataFrame(rev_rows)

In [58]:
# ── Data quality check ────────────────────────────────────────────
print(f"Current df_revenue shape: {df_revenue.shape}")
assert len(df_revenue) == 1, f"Expected 1 rows, got {len(df_revenue)}"
assert df_revenue.isnull().sum().sum() == 0, "Unexpected nulls found"
assert (df_revenue['total_mrr'] >= 0).all(), "Negative MRR found"
assert (df_revenue['new_mrr'] >= 0).all(), "Negative new MRR found"

Current df_revenue shape: (1, 5)


In [59]:
# ── Save ──────────────────────────────────────────────────────────
os.makedirs('data', exist_ok=True)
df_revenue.to_csv('data/revenue.csv', index=False)

In [60]:
# ── Summary ───────────────────────────────────────────────────────
final_mrr = df_revenue['total_mrr'].iloc[-1]
first_mrr = df_revenue['total_mrr'].iloc[0]
mrr_growth = ((final_mrr - first_mrr) / first_mrr) * 100
 
print("=" * 45)
print("revenue.csv — Quality Report")
print("=" * 45)
print(f"Rows             : {len(df_revenue)}")
print(f"Null values      : {df_revenue.isnull().sum().sum()}")
print(f"\nMRR Growth Story:")
print(df_revenue[['month', 'new_mrr', 'expansion_mrr', 'churned_mrr', 'total_mrr']].to_string(index=False))
print(f"\nKey metrics:")
print(f"  Starting MRR (Jan 2023) : ${first_mrr:,.2f}")
print(f"  Final MRR   (Dec 2024)  : ${final_mrr:,.2f}")
print(f"  MRR growth over 24 mo   : {mrr_growth:.1f}%")
print(f"  Implied ARR (Dec 2024)  : ${final_mrr * 12:,.2f}")
print("\n✅ Saved: data/revenue.csv")

revenue.csv — Quality Report
Rows             : 1
Null values      : 0

MRR Growth Story:
     month  new_mrr  expansion_mrr  churned_mrr  total_mrr
2024-12-01  45959.0         617.28       4611.0   41965.28

Key metrics:
  Starting MRR (Jan 2023) : $41,965.28
  Final MRR   (Dec 2024)  : $41,965.28
  MRR growth over 24 mo   : 0.0%
  Implied ARR (Dec 2024)  : $503,583.36

✅ Saved: data/revenue.csv
